# 第7周 · 周三：样本量与MDE——实验需要多少用户？

> 难度：⭐⭐⭐⭐ | 定位：掌握样本量计算和MDE的概念，能独立估算实验所需用户数和时长
> 前置依赖：W7D2 假设检验(α/β/Power) | 后续用途：实验设计、实验评审、资源规划
> 面试地位：大厂数据分析面试必考，尤其是MDE的计算和反推
> 配对课程：01python_learn W7D3（抽样与中心极限定理）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 为什么需要算样本量？——血的教训

"我们实验跑了3天，结果不显著。再跑几天看看？"

问题在于：如果不显著是因为样本量不够，等数据积累到足够多自然会显著。但如果策略真的没效果，等再久也不会显著。**不知道需要多少样本就开实验 = 蒙着眼睛开车。**

更危险的场景：实验跑了2周，p=0.03"显著"。PM要求上线。但你有没有想过——这个"显著"可能是Peeking了14次才碰到的？如果事前算过样本量，你知道需要3周的数据——那你就会在2周时坚持继续跑，避免一次可能的误报。

**样本量计算的三个价值**：
1. **诚实工具**：让业务方理解"以我们的流量，能检测到什么级别的效果"
2. **防Peeking**：事前定好终点，中间不偷看
3. **资源规划**：知道实验要跑多久，协调工程和运营资源

### 1.2 样本量的四个决定因素

```
n = f(α, β, σ, δ)

α（显著性水平）：通常0.05 → α越小，n越大
β（第二类错误率）：通常0.20（功效80%）→ β越小，n越大
σ（指标标准差）：指标波动越大，需要的样本越多
δ（MDE：最小可检测效应）：你想检测多小的效果？→ δ越小，n越大（指数级增长！）
```

**最关键的关系**：n ∝ 1/δ²。即MDE减半，样本量增长4倍。这就是为什么"精确检测小效应"如此昂贵。

### 1.3 MDE（Minimum Detectable Effect）——数据分析师的翻译工具

MDE = 在给定α、β、σ和样本量n下，能够被检测到的最小效应。

MDE是连接"统计世界"和"业务世界"的翻译层：
- 产品说："我想检测到2%的转化率提升"
- 你计算后说："那需要每组5万人，实验要跑3周"
- 产品说："太久了，我们只有1周的流量预算"
- 你说："那只能检测到4%以上的提升，低于4%的效果我们会'看不出来'——不是没效果，而是样本不够"

### 1.4 反向使用MDE——给定流量，能检测什么？

MDE可以正向计算（给定MDE→算样本量），也可以**反向计算**（给定样本量→算能检测的MDE）。反向计算在日常工作中其实更常用：

```
场景：老板说"我们下周要做个实验，只有5万用户"
你算一下MDE：每组2.5万人 → 能检测到的最小效应约3.5pp
结论：如果策略的效果小于3.5pp，我们可能检测不到
```

> **今天最重要的认知**：MDE是你的"诚实工具"——它让所有人理解"样本量有限的情况下，我们只能检测到足够大的效果"。小实验不能证明"没效果"，只能证明"没检测到大效果"。

### 1.5 面试高频考点

**面试官常问**：
- "样本量取决于哪些因素？各自如何影响？"
- "什么是MDE？MDE和样本量有什么关系？"
- "为什么MDE减半，样本量要增加4倍？"
- "有100万DAU，为什么不能做检测0.1%提升的实验？"

**回答要点**：
- 四个参数：α、β(Power)、σ(方差)、δ(MDE)
- MDE是在给定参数下能检测到的最小效应
- n ∝ 1/δ²（MDE减半→n×4）
- 0.1%的提升虽然"显著"但无业务价值，且需要上亿样本

## 🧩 区块2：知识点详细拆解

### 2.1 比例指标的样本量公式

**为什么重要**：转化率、留存率、点击率等比例指标是最常见的AB实验指标。

**公式（每组所需样本量）**：
```
n = (Z_{α/2}√[2p̄(1-p̄)] + Z_β√[p₁(1-p₁)+p₂(1-p₂)])² / (p₂-p₁)²

其中：
p₁ = 基线转化率
p₂ = p₁ + MDE（期望检测到的转化率）
p̄ = (p₁ + p₂) / 2
Z_{α/2} = 1.96（α=0.05时）
Z_β = 0.84（Power=80%时）
```

**关键特性**：
- 基线p越接近50%，所需样本量越大（方差最大）
- 基线p接近0或1时，所需样本量减小
- MDE越小，所需样本量指数级增长

### 2.2 连续指标的样本量公式

**为什么重要**：GMV、时长、人均订单数等连续指标也很常见。

**公式（每组所需样本量）**：
```
n = 2(Z_{α/2} + Z_β)² × σ² / δ²

其中：
σ² = 指标的方差
δ = MDE（绝对差值，不是相对比例）
```

**关键特性**：
- n ∝ σ²（方差越大，样本越多）
- n ∝ 1/δ²（MDE越小，样本越多）
- 与比例公式不同，连续指标的样本量与均值水平无关，只与方差有关

**降低连续指标方差的方法**：
- CUPED（用实验前数据做协变量调整）——可减少30-50%方差
- 分层抽样——确保关键维度均衡
- 截断/缩尾——处理极端异常值

### 2.3 MDE的四种表达方式

**为什么重要**：不同受众理解不同的MDE表达方式。

| 表达方式 | 公式 | 示例 | 适用受众 |
|---------|------|------|----------|
| 绝对差值 | δ = p₂ - p₁ | +2个百分点 | 分析师 |
| 相对提升 | δ_rel = (p₂-p₁)/p₁ | +20% | PM/运营 |
| 比值(Odds Ratio) | OR = p₂/(1-p₂) / (p₁/(1-p₁)) | 1.22 | 学术界 |
| 标准化效应 | Cohen's h | 0.065 | 跨实验比较 |

**翻译示例**：
- "MDE = 2pp绝对提升"（分析师）
- "能检测到20%以上的相对提升"（PM更好理解）
- "如果实验只能检测到2pp以上，那些1pp的提升可能被漏掉"（诚实沟通）

### 2.4 样本量计算器的Python实现

**为什么重要**：掌握底层实现，而不是依赖在线计算器。

```python
def sample_size_proportion(baseline, relative_lift, alpha=0.05, power=0.80):
    """计算比例指标每组所需样本量"""
    z_alpha = stats.norm.ppf(1 - alpha/2)  # 双侧
    z_beta = stats.norm.ppf(power)
    
    p1 = baseline
    p2 = baseline * (1 + relative_lift)  # 相对提升
    p_bar = (p1 + p2) / 2
    
    numerator = (z_alpha * np.sqrt(2*p_bar*(1-p_bar)) + 
                 z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2
    denominator = (p2 - p1)**2
    
    return int(np.ceil(numerator / denominator))
```

### 2.5 样本量与实际流量——从n到实验天数

**为什么重要**：样本量计算只是第一步，还需要转化成实验时长。

**计算步骤**：
```
1. 计算每组所需样本量 n
2. 确定可用的日活用户数 DAU
3. 考虑分流比例（通常50:50，也可能是其他比例）
4. 考虑留存窗口（如测7留需要等7天）
5. 实验天数 = n / (DAU × 分流比例) + 指标窗口
```

**示例**：
```
n = 35,000/组（检测2%相对提升）
DAU = 10,000（可进入实验的新用户/天）
分流比例 = 50%
指标窗口 = 0（当天转化）

实验天数 = 35,000 / (10,000 × 0.5) = 7天
两组共需：35,000 × 2 = 70,000新用户
```

**考虑留存窗口**：
```
如果主要指标是7日留存，用户进入实验后需要等7天才能看到结果
实际实验天数 = 积累天数 + 7天窗口
           = 7天 + 7天 = 14天
```

### 2.6 样本量不足时的选择——五个策略

**为什么重要**：流量不够的时候，不是只有"取消实验"一个选项。

| 策略 | 代价 | 适用场景 |
|------|------|----------|
| 1. 接受更大的MDE | 漏掉小效应 | 策略预期效应大的场景 |
| 2. 放宽α（如0.05→0.10） | 更多误报 | 低风险实验 |
| 3. 降低Power（如80%→70%） | 更多漏报 | 探索性实验 |
| 4. 改变分流比例（如50:50→80:20） | 实验组样本增大但对照组变小 | 确定方向后加速积累 |
| 5. 放宽随机化单元（如用户→会话） | SUTVA问题 | 短期UI实验 |

**最重要的判断**：如果流量太少的根本原因是"DAU就这么多"，最诚实的做法是**接受更大的MDE**。与其假装能检测到1%的提升，不如诚实地说"我们只能检测到5%以上的效果"。

### 2.7 为什么n ∝ 1/δ²？——直觉解释

**为什么重要**：理解这个关系比记住公式更重要。

标准误 SE = σ/√n。要检测到δ大小的效应，信号需要足够大于噪声：
```
δ / (σ/√n) > Z_{α/2} + Z_β
```
移项得到：
```
√n > (Z_{α/2} + Z_β) × σ / δ
n > (Z_{α/2} + Z_β)² × σ² / δ²
```
n和δ²成反比。δ变小一半 → n需要4倍。

**直观理解**：你要在一片噪音中辨认一个信号。信号越弱（δ越小），你需要越多的观测才能确信它不是噪声。信号弱一半，样本需要4倍——因为不确定性以1/√n的速度下降。

### 2.8 业界样本量速查表

**为什么重要**：在实际工作中快速估算，不需要每次都精确计算。

**比例指标速查（α=0.05, Power=80%，双侧）**：

| 基线 | MDE(绝对) | 每组样本量 | MDE(绝对) | 每组样本量 |
|------|-----------|-----------|-----------|-----------|
| 5% | +1pp | 7,200 | +0.5pp | 28,500 |
| 10% | +1pp | 12,500 | +2pp | 3,200 |
| 20% | +1pp | 20,000 | +2pp | 5,100 |
| 50% | +1pp | 25,000 | +2pp | 6,400 |

**记忆口诀**：
- 10%基线+2pp → 约3000/组（常见场景）
- 10%基线+1pp → 约12000/组（4倍关系！）
- 5%基线+0.5pp → 约28000/组（低基线+小效应=贵）

## 💻 区块3：动手练习

### 步骤0：样本量计算器

实现一个完整的样本量计算器，支持比例指标和连续指标。

In [ ]:
import numpy as np
from scipy import stats

# 步骤0：实现样本量计算器
def sample_size_proportion(baseline, relative_lift, alpha=0.05, power=0.80):
    """计算比例指标每组所需样本量（相对提升方式）"""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    p1 = baseline
    p2 = baseline * (1 + relative_lift)
    p_bar = (p1 + p2) / 2
    num = (z_alpha * np.sqrt(2*p_bar*(1-p_bar)) +
           z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2
    den = (p2 - p1)**2
    return int(np.ceil(num / den))

def sample_size_proportion_absolute(baseline, absolute_lift, alpha=0.05, power=0.80):
    """计算比例指标每组所需样本量（绝对提升方式）"""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    p1 = baseline
    p2 = baseline + absolute_lift
    p_bar = (p1 + p2) / 2
    num = (z_alpha * np.sqrt(2*p_bar*(1-p_bar)) +
           z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2
    den = (p2 - p1)**2
    return int(np.ceil(num / den))

def sample_size_continuous(std, mde, alpha=0.05, power=0.80):
    """计算连续指标每组所需样本量"""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    n = 2 * (z_alpha + z_beta)**2 * std**2 / mde**2
    return int(np.ceil(n))

def mde_from_sample_size(baseline, n_per_group, alpha=0.05, power=0.80):
    """反向计算：给定样本量，能检测到的最小相对提升"""
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    # 使用近似公式反推
    p1 = baseline
    factor = (z_alpha + z_beta) / np.sqrt(n_per_group)
    # MDE ≈ factor * sqrt(2 * p1 * (1-p1))
    mde_absolute = factor * np.sqrt(2 * p1 * (1 - p1))
    mde_relative = mde_absolute / baseline
    return mde_absolute, mde_relative

print("样本量计算器已就绪")
print("支持：比例指标(绝对值/相对值)、连续指标、反向MDE计算")

### 步骤1：基础练习——MDE与样本量的关系

In [ ]:
print("="*60)
print("步骤1.1：MDE与样本量——指数级关系")
print("="*60)

baseline = 0.10  # 基线转化率10%

print(f"基线转化率: {baseline:.0%}, α=0.05, Power=80%")
print(f"\n{'相对提升':<12} {'绝对提升':<12} {'每组样本量':<15} {'两组共需':<15}")
print("-" * 54)

for relative_lift in [0.30, 0.20, 0.15, 0.10, 0.05, 0.02, 0.01]:
    n = sample_size_proportion(baseline, relative_lift)
    abs_lift = baseline * relative_lift
    print(f"{relative_lift:<12.0%} {abs_lift:<12.1%} {n:<15,} {n*2:<15,}")

print(f"\n关键洞察:")
print(f"  检测30%提升 → 每组约300人")
print(f"  检测10%提升 → 每组约3,000人（MDE×1/3，样本×10）")
print(f"  检测2%提升  → 每组约35,000人（MDE×1/5，样本×100+）")
print(f"  → MDE越小，样本量呈指数级增长（n ∝ 1/MDE²）")

In [ ]:
print("="*60)
print("步骤1.2：基线水平对样本量的影响")
print("="*60)

relative_lift = 0.10  # 固定10%相对提升

print(f"固定相对提升: {relative_lift:.0%}, α=0.05, Power=80%")
print(f"\n{'基线转化率':<12} {'绝对提升':<12} {'每组样本量':<15}")
print("-" * 39)

for baseline in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n = sample_size_proportion(baseline, relative_lift)
    abs_lift = baseline * relative_lift
    print(f"{baseline:<12.0%} {abs_lift:<12.1%} {n:<15,}")

print(f"\n→ 基线越接近50%，样本量越大（因为方差p(1-p)在p=0.5时最大）")
print(f"→ 基线5% vs 50%：同样的相对提升，样本量差约1.7倍")

In [ ]:
print("="*60)
print("步骤1.3：α和Power对样本量的影响")
print("="*60)

baseline = 0.10
mde_rel = 0.10

print(f"基线={baseline:.0%}, MDE={mde_rel:.0%}相对提升")
print(f"\n{'α':<6} {'Power':<8} {'每组样本量':<15}")
print("-" * 29)

for alpha in [0.01, 0.05, 0.10]:
    for power in [0.70, 0.80, 0.90]:
        n = sample_size_proportion(baseline, mde_rel, alpha, power)
        print(f"{alpha:<6.2f} {power:<8.0%} {n:<15,}")

print(f"\n→ α越小（更严格），样本量越大")
print(f"→ Power越高（不想漏报），样本量越大")
print(f"→ α=0.01+Power=90% vs α=0.05+Power=70%：样本量相差约2.5倍")

### 步骤2：进阶练习——反向计算MDE和实验天数

In [ ]:
print("="*60)
print("步骤2.1：反向计算MDE——给定流量，能检测什么？")
print("="*60)

print("场景：老板说'下周做实验，每天约5000新用户'")
print()

dau_new_users = 5000
days_available = 7
traffic_split = 0.5  # 50:50分流

n_per_group = int(dau_new_users * days_available * traffic_split)
print(f"可用新用户: {dau_new_users}/天 × {days_available}天 = {dau_new_users * days_available}人")
print(f"每组可得: {n_per_group}人（{traffic_split:.0%}分流）")

print(f"\n不同基线下的可检测MDE:")
print(f"{'基线':<10} {'可检测绝对MDE':<18} {'可检测相对MDE':<15}")
print("-" * 43)

for baseline in [0.05, 0.10, 0.20, 0.30]:
    mde_abs, mde_rel = mde_from_sample_size(baseline, n_per_group)
    print(f"{baseline:<10.0%} {mde_abs:<18.2%} {mde_rel:<15.1%}")

print(f"\n→ 如果基线10%，只能检测到约{MDE_REL*100 if 'MDE_REL' in dir() else 'X'}%以上的相对提升")
print(f"→ 低于此阈值的效果，即使真实存在，也可能检测不到")

In [ ]:
print("="*60)
print("步骤2.2：考虑留存窗口的实验时长计算")
print("="*60)

def experiment_days(n_per_group, dau, traffic_split, metric_window_days=0):
    """计算实验需要运行的天数"""
    accumulation_days = np.ceil(n_per_group / (dau * traffic_split))
    total_days = accumulation_days + metric_window_days
    return int(accumulation_days), int(total_days)

print("场景1：主要指标=当天转化率（无窗口期）")
n = sample_size_proportion(0.10, 0.10)
acc_days, total = experiment_days(n, 5000, 0.5, 0)
print(f"  需要{n:,}/组 → 积累{acc_days}天 → 实验共{total}天")

print("\n场景2：主要指标=7日留存（7天窗口期）")
n = sample_size_proportion(0.25, 0.10)  # 25%基线7留
acc_days, total = experiment_days(n, 3000, 0.5, 7)
print(f"  需要{n:,}/组 → 积累{acc_days}天+7天窗口 → 实验共{total}天")

print("\n场景3：主要指标=30日留存（30天窗口期）")
n = sample_size_proportion(0.15, 0.15)  # 15%基线30留
acc_days, total = experiment_days(n, 3000, 0.5, 30)
print(f"  需要{n:,}/组 → 积累{acc_days}天+30天窗口 → 实验共{total}天")
print(f"  → 30日留存实验需要1.5个月，几乎不可行！")
print(f"  → 实践中常用7留或14留作为30留的代理指标")

### 步骤3：挑战练习——连续指标的样本量

3.1 计算GMV（连续指标）的样本量
3.2 对比CUPED方差缩减前后的样本量差异

In [ ]:
print("="*60)
print("步骤3：连续指标样本量——GMV实验")
print("="*60)

np.random.seed(42)

# 模拟GMV数据（连续指标）
n_sim = 10000
baseline_gmv_mean = 200  # 人均GMV 200元
baseline_gmv_std = 500    # 标准差500元（GMV通常高度偏态）

# 生成偏态GMV数据
gmv_data = np.random.lognormal(
    mean=np.log(baseline_gmv_mean**2 / np.sqrt(baseline_gmv_mean**2 + baseline_gmv_std**2)),
    sigma=np.sqrt(np.log(1 + baseline_gmv_std**2 / baseline_gmv_mean**2)),
    size=n_sim
)

print(f"模拟GMV数据: 均值={gmv_data.mean():.1f}, 标准差={gmv_data.std():.1f}")
print(f"变异系数CV = {gmv_data.std()/gmv_data.mean():.2f}")

# 计算不同MDE下的样本量
print(f"\n{'MDE(元)':<12} {'MDE(相对)':<12} {'每组样本量':<15} {'两组共需':<15}")
print("-" * 54)

for mde_rel in [0.20, 0.15, 0.10, 0.05, 0.03, 0.02]:
    mde_abs = baseline_gmv_mean * mde_rel
    n = sample_size_continuous(baseline_gmv_std, mde_abs)
    print(f"{mde_abs:<12.1f} {mde_rel:<12.0%} {n:<15,} {n*2:<15,}")

print(f"\n→ GMV的变异系数大（{gmv_data.std()/gmv_data.mean():.2f}），需要的样本量远大于比例指标")
print(f"→ 这就是为什么需要用CUPED等技术减少方差")

# 模拟CUPED效果
print(f"\nCUPED方差缩减效果模拟:")
correlation_with_pre = 0.7  # 实验前GMV与实验期GMV的相关系数
variance_reduction = correlation_with_pre ** 2
reduced_std = baseline_gmv_std * np.sqrt(1 - variance_reduction)

print(f"  实验前-实验期GMV相关系数: {correlation_with_pre}")
print(f"  方差缩减比例: {variance_reduction:.0%}")
print(f"  原始标准差: {baseline_gmv_std:.0f}")
print(f"  缩减后标准差: {reduced_std:.0f}")

mde = baseline_gmv_mean * 0.05
n_original = sample_size_continuous(baseline_gmv_std, mde)
n_cuped = sample_size_continuous(reduced_std, mde)
print(f"\n  检测5%提升所需样本量:")
print(f"    无CUPED: {n_original:,}/组")
print(f"    有CUPED: {n_cuped:,}/组")
print(f"    节省: {(1-n_cuped/n_original)*100:.0f}%")


## 📊 区块4：业务场景应用

### 场景1：产品说"我想检测1%的提升"——MDE的诚实沟通

**问题**：PM想测试新的注册流程，期望能检测到相对1%的转化率提升。目前注册转化率基线是20%，每天约3000新用户。

**分析**：
```
基线=20%, MDE_relative=1%, α=0.05, Power=80%
所需样本量 → 每组约1,400,000人！
每天3000新用户 × 50%分流 = 1500/天/组
实验天数 = 1,400,000 / 1500 ≈ 933天 ≈ 2.5年！
```

**沟通策略**：
1. 展示计算过程，让PM理解这不是"我们不想做"，而是"数学上不可行"
2. 提供替代方案：如果接受5%的MDE（相对），只需约14,800/组 → 约10天
3. 讨论：1%的提升真的有业务价值吗？如果20%→20.2%，ROI可能很低

**决策**：接受更大的MDE（5%），或者放弃这个实验，把流量留给更有潜力的策略。

### 场景2：实验平台说"我们有100万DAU，什么实验都能做"

**问题**：大厂的实验平台拥有海量用户，但真的"什么实验都能做"吗？

**分析**：
1. 100万DAU看起来很多，但：
   - 可能只有10%是新用户（10万/天）
   - 可能有50个实验同时运行分流
   - 可能只有特定城市/平台的用户可用
2. 即使流量无限，你也面临：
   - 微小效应的"显著但无价值"困境
   - 实验间相互干扰（同一用户进入多个实验）
   - 组织注意力有限（不能同时消化太多实验结论）

**决策**：大厂不是"什么实验都能做"，而是"能做更多实验"。样本量计算仍然是必要的——它帮助判断MDE和业务价值是否匹配。

### 场景3：改变分流比例——什么时候不该用50:50

**问题**：某App想测试一个新功能，PM担心新功能可能有负面影响，希望只有10%的用户进入实验组。

**分析**：
- 分流比例从50:50改为90:10 → 实验组样本量减少5倍
- 如果需要每组5000人，实验组只有1000人 → Power下降
- 但可以通过延长实验时间来补偿

**权衡**：
- 50:50：统计效率最高（相同总样本下Power最大）
- 90:10：更安全（只有10%用户暴露于风险），但需要更长时间
- 最佳实践：不确定安全性时先用10%→验证安全后扩大到50%

**决策**：先10%流量跑1周验证安全性（护栏指标），确认安全后扩大到50%评估效果。

### 场景4：为什么竞争对手能测而我们不行？——流量差异的诚实评估

**问题**：老板说"某大厂每周能跑100个实验，我们为什么跑不了？"

**分析**：这本质是流量焦虑。大厂DAU是你的100倍，能检测的MDE是你的1/10。

**应对框架**：
1. 算MDE：以你的流量水平，能检测到什么级别的效果
2. 提策略：与其跑很多"小MDE"实验，不如聚焦"大MDE"的高潜力策略
3. 做取舍：不是所有决策都需要AB实验——有些事情方向足够明确可以直接做
4. 建能力：投资长期实验基础设施，流量会随着用户增长而增长

**核心认知**：流量少不是劣势，而是约束条件。在约束下做最优决策，本身就是数据分析师的价值。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能说出样本量计算的四个参数
- [ ] 能解释为什么MDE越小样本量越大
- [ ] 能用在线计算器算出常见场景的样本量
- [ ] 能区分比例指标和连续指标的样本量公式

**答案提示**：
- 四个参数：α(P(误报))、β(P(漏报))、σ(方差)、δ(MDE)
- n ∝ 1/MDE²：MDE减半，样本量4倍
- 比例指标取决于基线p，连续指标取决于方差σ²

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能手动计算比例指标的样本量（查表或公式）
- [ ] 能反向计算MDE（给定样本量→可检测效果）
- [ ] 能将样本量转换为实验天数
- [ ] 能解释为什么基线50%时样本量最大

**答案提示**：
- 比例公式：n = (Zα√[2p̄(1-p̄)] + Zβ√[p₁(1-p₁)+p₂(1-p₂)])² / (p₂-p₁)²
- 反向MDE：MDE ≈ (Zα+Zβ)×√[2p(1-p)/n]
- 实验天数 = n/(DAU×分流比) + 指标窗口
- p=50%时方差p(1-p)=0.25最大

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能在流量不足时提出替代策略（改变α/改变分流比/接受更大MDE）
- [ ] 能解释CUPED如何减少所需样本量
- [ ] 能为公司的实验平台设计样本量计算器的产品需求
- [ ] 能评估"用代理指标减少实验时间"的可行性

**答案提示**：
- 替代策略：增大MDE/放宽α/改变分流/用代理指标
- CUPED：用实验前数据协变量调整，方差缩减比例=R²
- 产品需求：输入基线+MDE→输出样本量+天数+流量预算
- 代理指标：用短期敏感指标（如点击率）替代长期指标（如留存）

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W7D2 假设检验 | 样本量公式依赖α/β/Power参数 |
| **← 前置依赖** | W1 指标体系 | 不同指标类型（比例/连续）用不同公式 |
| **→ 后续用途** | W7D4 实验设计全流程 | 样本量计算是实验设计的第二步 |
| **→ 后续用途** | W7D5 Peeking | 事前确定样本量是防止Peeking的基础 |
| **→ 后续用途** | W8 CUPED | 方差缩减技术直接减少所需样本量 |
| **业务连接** | 所有实验的资源规划 | 样本量→实验天数→工程排期→运营配合 |

### 横向连接：与其他分析方法的关联

**与工程系统的连接**：
- 实验平台的流量分配系统
- 分流比例的动态调整机制
- 实验预算管理和流量配额

**与项目管理的连接**：
- 样本量→实验时长→项目排期
- 多个实验的流量竞争和优先级排序
- 实验队列管理（避免用户同时进入太多实验）

## 📝 区块7：当日小结

> **样本量计算是实验之前必须做的功课。四个参数：α(显著性)、β(功效)、σ(方差)、δ(MDE)。最重要的关系：n ∝ 1/δ²——MDE减半，样本量增长4倍。MDE是你的诚实工具——它让业务方理解样本量有限的情况下只能检测到足够大的效果。比例指标和连续指标有各自的样本量公式。样本量可以正向计算（给定MDE→求n）也可以反向计算（给定n→求可检测MDE）。CUPED等方差缩减技术可以大幅减少所需样本量。**

**三条金句带回家**：
1. MDE是数据分析师的"诚实工具"——小实验不能证明"没效果"，只能证明"没检测到大效果"
2. MDE减半，样本量×4——这就是为什么"精确检测小效应"如此昂贵
3. 不知道需要多少样本就开实验 = 蒙着眼睛开车

### 🔖 课后拓展

1. **练习**：为你公司最近做的一个AB实验计算样本量。输入基线指标、期望MDE、日活流量，计算所需实验天数，与实际的实验时长对比。

2. **思考**：如果一个实验的MDE=2%，但策略上线后只产生了1.5%的提升（不显著），这个策略"没有效果"吗？怎么理解这个结果？

3. **预习**：明天要学的实验设计完整流程——从假设到决策的9个步骤。

4. **面试准备**：背诵以下面试题答案：
   - "样本量取决于哪些参数？各自如何影响？"
   - "什么是MDE？为什么它重要？"
   - "为什么检测1%的提升比检测5%的提升难25倍？"（因为n∝1/δ²，5倍MDE→25倍样本）

---

*03analyst_thinking · Week 7 Day 3 · 2026-05-21*